# fase_2 - script_afrida Migration

This notebook handles migration of database from old DB to new DB for fase 2.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


## 2. Ambil Data dari DB Lama

In [4]:
def cek_kualitas(df, nama_tabel, pk_col):
    print(f"\n{'='*50}")
    print(f"  CEK DATA: {nama_tabel}")
    print(f"{'='*50}")
    print("Missing values:")
    print(df.isnull().sum())
    print(f"\nDuplikat {pk_col}: {df[pk_col].duplicated().sum()} baris")
    print("\nSample data (5 baris pertama):")
    
    display(df.head())

In [9]:
df_periode_lama = pd.read_sql('SELECT * FROM periode', db_old)
cek_kualitas(df_periode_lama, 'periode (lama)', 'idperiode')

print(f"Jumlah data awal: {len(df_periode_lama)}")


  CEK DATA: periode (lama)
Missing values:
idperiode        0
nama_term        0
tanggal          0
bulan_awal       0
tahun_awal       0
idpendkursus     0
jml_sesi         0
tahun_ajar       0
mitra           93
dtype: int64

Duplikat idperiode: 0 baris

Sample data (5 baris pertama):


,idperiode,nama_term,tanggal,bulan_awal,tahun_awal,idpendkursus,jml_sesi,tahun_ajar,mitra
0,P00006,General English Term I July-October 2023,4.0,Juli,2023.0,K00001,30.0,2023/2024,None
1,P00008,General English Term II Oct '23 - Feb '24,25.0,Oktober,2023.0,K00001,30.0,2023/2024,None
2,P00009,General English Term III Feb-Jun 2024,21.0,Februari,2024.0,K00001,30.0,2023/2024,None
3,P00010,Coding Semester I-2023/2024,1.0,Agustus,2023.0,K00002,18.0,2023/2024,None
4,P00011,Coding Semester II-2023/2024,23.0,Januari,2024.0,K00002,18.0,2023/2024,None


Jumlah data awal: 93


In [14]:
# =================================================
# PROSES TABEL: periode
# =================================================

# 2. Mapping nama bulan Indonesia → angka
bulan_map = {
    'Januari': 1, 'Februari': 2, 'Maret': 3,
    'April': 4, 'Mei': 5, 'Juni': 6,
    'Juli': 7, 'Agustus': 8, 'September': 9,
    'Oktober': 10, 'November': 11, 'Desember': 12
}

# 3. Konstruksi tanggal_mulai dari tanggal, bulan_awal, tahun_awal
def buat_tanggal(row):
    try:
        hari = int(row['tanggal'])
        bulan = bulan_map[row['bulan_awal']]
        tahun = int(row['tahun_awal'])
        # Gabung jadi string 'YYYY-MM-DD' lalu parse
        date_str = f"{tahun}-{bulan:02d}-{hari:02d}"
        return pd.to_datetime(date_str).date()
    except:
        # Jika ada yang kosong atau error, isi dengan NaT (kita bisa investigasi lebih lanjut)
        return pd.NaT

df_periode_lama['tanggal_mulai'] = df_periode_lama.apply(buat_tanggal, axis=1)

# 4. Hapus kolom lama yang tidak diperlukan
kolom_dibuang = ['tanggal', 'bulan_awal', 'tahun_awal', 'mitra']
df_periode = df_periode_lama.drop(columns=kolom_dibuang, errors='ignore')

# 5. Rename kolom-kolom yang tersisa
mapping_periode = {
    'idperiode': 'id_periode',
    'nama_term': 'nama_periode',
    'idpendkursus': 'id_kursus',
    'jml_sesi': 'jumlah_sesi',
    'tahun_ajar': 'tahun_ajar'        # biarkan sama dulu
}
df_periode = df_periode.rename(columns=mapping_periode)

# 6. Tambah kolom baru
df_periode['status'] = 1
df_periode['is_active'] = 1

# 7. Pastikan tipe data
df_periode['id_periode'] = df_periode['id_periode'].astype(str)  # kode seperti 'P00006', jadi str
df_periode['id_kursus'] = df_periode['id_kursus'].astype(str)    # jika di DB baru juga varchar, sesuaikan
df_periode['jumlah_sesi'] = df_periode['jumlah_sesi'].astype(int)
df_periode['tahun_ajar'] = df_periode['tahun_ajar'].astype(str)
# tanggal_mulai sudah bertipe date

# 8. Cek kualitas
cek_kualitas(df_periode, 'periode', 'id_periode')

# Cek tanggal yang gagal di-parse (NaT)
gagal = df_periode[df_periode['tanggal_mulai'].isna()]
if len(gagal) > 0:
    print(f"\n⚠️  {len(gagal)} baris gagal membuat tanggal_mulai:")
    print(gagal[['id_periode', 'nama_periode']])

# Cek FK ke tabel kursus di db_old (karena db_new belum ada isinya)
try:
    # Asumsi di db_old tabel kursus punya primary key 'idpendkursus'
    kursus_old = pd.read_sql('SELECT idpendkursus FROM pendidikankursus', db_old)
    id_kursus_set = set(kursus_old['idpendkursus'].unique())
    missing_fk = df_periode[~df_periode['id_kursus'].isin(id_kursus_set)]
    if len(missing_fk) > 0:
        print(f"\n⚠️  Ditemukan {len(missing_fk)} id_kursus di periode yang TIDAK ada di tabel kursus (db_old):")
        print(missing_fk[['id_periode', 'id_kursus']].head())
    else:
        print("✅ Semua id_kursus di periode valid (ditemukan di db_old).")
except Exception as e:
    print(f"ℹ️  Gagal validasi FK ke kursus via db_old: {e}")


  CEK DATA: periode
Missing values:
id_periode       0
nama_periode     0
id_kursus        0
jumlah_sesi      0
tahun_ajar       0
tanggal_mulai    0
status           0
is_active        0
dtype: int64

Duplikat id_periode: 0 baris

Sample data (5 baris pertama):


,id_periode,nama_periode,id_kursus,jumlah_sesi,tahun_ajar,tanggal_mulai,status,is_active
0,P00006,General English Term I July-October 2023,K00001,30,2023/2024,2023-07-04,1,1
1,P00008,General English Term II Oct '23 - Feb '24,K00001,30,2023/2024,2023-10-25,1,1
2,P00009,General English Term III Feb-Jun 2024,K00001,30,2023/2024,2024-02-21,1,1
3,P00010,Coding Semester I-2023/2024,K00002,18,2023/2024,2023-08-01,1,1
4,P00011,Coding Semester II-2023/2024,K00002,18,2023/2024,2024-01-23,1,1


✅ Semua id_kursus di periode valid (ditemukan di db_old).


In [15]:
# =================================================
# PROSES TABEL: parameter_nilai
# =================================================

# 1. Tarik data
df_param_lama = pd.read_sql('SELECT * FROM parameter_nilai', db_old)
print(f"Jumlah data awal: {len(df_param_lama)}")
print("Kolom:", df_param_lama.columns.tolist())
display(df_param_lama.head(10))

# 2. Cek tipe data & missing
print("\nInfo:")
print(df_param_lama.info())
print("\nMissing values:")
print(df_param_lama.isnull().sum())
print("\nNilai unik isnumber:")
print(df_param_lama['isnumber'].unique())
print("\nNilai unik idlevel:")
print(df_param_lama['idlevel'].unique())

Jumlah data awal: 1187
Kolom: ['idp_nilai', 'idlevel', 'parameter', 'isnumber']


,idp_nilai,idlevel,parameter,isnumber
0,P00745,L00022,Class participation,0.0
1,P00746,L00022,Oral,0.0
2,P00747,L00022,Listening,0.0
3,P00748,L00022,Writing,0.0
4,P00749,L00022,Writing-1,1.0
5,P00750,L00022,Grammar Reading-1,1.0
6,P00751,L00022,Presentation-1,1.0
7,P00752,L00022,Listening-1,1.0
8,P00753,L00022,Writing-2,1.0
9,P00754,L00022,Grammar Reading-2,1.0



Info:
<class 'pandas.DataFrame'>
RangeIndex: 1187 entries, 0 to 1186
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   idp_nilai  1187 non-null   str    
 1   idlevel    1187 non-null   str    
 2   parameter  1187 non-null   str    
 3   isnumber   1187 non-null   float64
dtypes: float64(1), str(3)
memory usage: 37.2 KB
None

Missing values:
idp_nilai    0
idlevel      0
parameter    0
isnumber     0
dtype: int64

Nilai unik isnumber:
[0. 1.]

Nilai unik idlevel:
<StringArray>
['L00022', 'L00011', 'L00001', 'L00045', 'L00047', 'L00002', 'L00003',
 'L00004', 'L00005', 'L00006',
 ...
 'L00168', 'L00169', 'L00170', 'L00171', 'L00172', 'L00173', 'L00174',
 'L00158', 'L00184', 'L00185']
Length: 142, dtype: str


In [17]:

# 2. Mapping kolom
mapping_param = {
    'idp_nilai': 'id_parameter_nilai',
    'idlevel': 'id_level',
    'parameter': 'nama_parameter',
    'isnumber': 'status_parameter'
}
df_param = df_param_lama.rename(columns=mapping_param)

# 3. Konversi tipe data
df_param['id_parameter_nilai'] = df_param['id_parameter_nilai'].astype(str)   # tetap VARCHAR
df_param['id_level'] = df_param['id_level'].astype(str)                       # VARCHAR juga
df_param['nama_parameter'] = df_param['nama_parameter'].astype(str)
# status_parameter: float -> int (sudah 0.0 / 1.0, aman dikonversi)
df_param['status_parameter'] = df_param['status_parameter'].astype(int)      # jadi tinyint

# 4. Cek kualitas
cek_kualitas(df_param, 'parameter_nilai', 'id_parameter_nilai')

# 5. Validasi Foreign Key `id_level` ke tabel `level` di db_old
try:
    level_old = pd.read_sql('SELECT idlevel FROM level', db_old)  # sesuaikan nama kolom PK level
    id_level_set = set(level_old['idlevel'].unique())
    missing_fk = df_param[~df_param['id_level'].isin(id_level_set)]
    if len(missing_fk) > 0:
        print(f"\n⚠️  Ditemukan {len(missing_fk)} id_level di parameter_nilai yang TIDAK ada di tabel level (db_old):")
        print(missing_fk[['id_parameter_nilai', 'id_level']].head())
    else:
        print("✅ Semua id_level di parameter_nilai valid (ditemukan di db_old).")
except Exception as e:
    print(f"ℹ️  Gagal validasi FK ke level via db_old: {e}")
    print("   Lanjut tanpa validasi FK. Pastikan tabel level sudah ada nanti.")



  CEK DATA: parameter_nilai
Missing values:
id_parameter_nilai    0
id_level              0
nama_parameter        0
status_parameter      0
dtype: int64

Duplikat id_parameter_nilai: 0 baris

Sample data (5 baris pertama):


,id_parameter_nilai,id_level,nama_parameter,status_parameter
0,P00745,L00022,Class participation,0
1,P00746,L00022,Oral,0
2,P00747,L00022,Listening,0
3,P00748,L00022,Writing,0
4,P00749,L00022,Writing-1,1


✅ Semua id_level di parameter_nilai valid (ditemukan di db_old).


## 3. Transform Data (jika diperlukan)

In [ ]:
# TODO: Tambahkan transformasi data di sini jika diperlukan
# Contoh: rename columns, convert data types, handle missing values, etc.

df = pd.DataFrame(data_old)
print(f"Data shape: {{df.shape}}")
print(f"Columns: {{df.columns.tolist()}}")

## 4. Insert ke DB Baru

In [ ]:
# TODO: Buat insert query sesuai dengan struktur tabel baru
insert_query = """INSERT INTO [NEW_TABLE_NAME] (col1, col2, col3) VALUES (%s, %s, %s)"""

try:
    for record in data_old:
        # TODO: Map columns dari DB lama ke DB baru
        cursor_new.execute(insert_query, (record['col1'], record['col2'], record['col3']))
    
    db_new.commit()
    print(f"Successfully inserted {{len(data_old)}} records to new DB")
except Exception as e:
    print(f"Error: {{e}}")
    db_new.rollback()

## 5. Verifikasi Data

In [ ]:
# Verify data di DB baru
try:
    cursor_new.execute("SELECT COUNT(*) as count FROM [NEW_TABLE_NAME]")
    result = cursor_new.fetchone()
    count_new = result['count']
except:
    count_new = len(data_old)  # Fallback jika query gagal

print(f"Total records from old DB: {{len(data_old)}}")
print(f"Total records in new DB: {{count_new}}")

if count_new == len(data_old):
    print("✓ Verifikasi OK - Jumlah record cocok")
else:
    print(f"⚠ Warning - Perbedaan: {{abs(count_new - len(data_old))}} record")

## 6. Return Hasil Migrasi untuk migrate_db.py

In [ ]:
import json
from datetime import datetime

# Create migration result yang akan dikumpulkan oleh migrate_db.py
migration_result = {{
    'fase': 'fase_2',
    'script': 'script_afrida',
    'fase_num': 2,
    'status': 'completed',
    'records_migrated': len(data_old),
    'records_in_new_db': count_new,
    'verified': count_new == len(data_old),
    'timestamp': datetime.now().isoformat(),
    'message': 'Migrasi tabel [NAMA TABEL] selesai'
}}

print("\n" + "="*60)
print("HASIL MIGRASI - fase_2 / script_afrida")
print("="*60)
print(json.dumps(migration_result, indent=2))
print("="*60)

## 7. Close Connection

In [ ]:
# Close semua koneksi database
try:
    cursor_old.close()
    cursor_new.close()
    db_old.close()
    db_new.close()
    print("✓ Database connections closed")
except:
    print("⚠ Error closing connections (mungkin sudah tertutup)")